In [1]:
import xgboost as xgb
import matplotlib.pyplot as plt
import os

artifact_dir = r"C:\Users\loq\.gemini\antigravity\brain\08f08955-7051-4470-bcba-a2b3a9aebbfd"

models = {
    "Daily_Long": "models/daily_xgb/xgb_long_model.json",
    "Daily_Short": "models/daily_xgb/xgb_short_model.json",
    "1Hour_Long": "models/v8_upstox_3y/xgb_long_model.json",
    "1Hour_Short": "models/v8_upstox_3y/xgb_short_model.json",
    "30Min_Long": "models/v1_30min/xgb_long_model.json",
    "30Min_Short": "models/v1_30min/xgb_short_model.json",
    "15Min_Long": "models/v1_15min/xgb_long_model.json",
    "15Min_Short": "models/v1_15min/xgb_short_model.json"
}

os.makedirs(artifact_dir, exist_ok=True)

for name, path in models.items():
    if not os.path.exists(path):
        print(f"Model file not found: {path}")
        continue
    
    try:
        model = xgb.Booster()
        model.load_model(path)
        
        # Plot Feature Importance (Weight)
        plt.figure(figsize=(10, 8))
        ax = xgb.plot_importance(model, max_num_features=20, importance_type='weight', title=f"Feature Importance (Weight) - {name}")
        plt.tight_layout()
        plt.savefig(os.path.join(artifact_dir, f"{name}_importance_weight.png"), dpi=300, bbox_inches='tight')
        plt.close('all')
        
        # Plot Feature Importance (Gain)
        plt.figure(figsize=(10, 8))
        ax = xgb.plot_importance(model, max_num_features=20, importance_type='gain', title=f"Feature Importance (Gain) - {name}")
        plt.tight_layout()
        plt.savefig(os.path.join(artifact_dir, f"{name}_importance_gain.png"), dpi=300, bbox_inches='tight')
        plt.close('all')
        
        print(f"Generated plots for {name}")
    except Exception as e:
        print(f"Error processing {name}: {e}")


Generated plots for Daily_Long


Generated plots for Daily_Short


Generated plots for 1Hour_Long


Generated plots for 1Hour_Short


Generated plots for 30Min_Long


Generated plots for 30Min_Short


Generated plots for 15Min_Long


Generated plots for 15Min_Short


In [2]:
!uv pip install shap

Resolved 18 packages in 16.37s
Prepared 10 packages in 46.18s
Installed 18 packages in 753ms
 + cloudpickle==3.1.2
 + colorama==0.4.6
 + joblib==1.5.3
 + llvmlite==0.47.0
 + narwhals==2.22.0
 + numba==0.65.1
 + numpy==2.4.6
 + packaging==26.2
 + pandas==3.0.3
 + python-dateutil==2.9.0.post0
 + scikit-learn==1.9.0
 + scipy==1.17.1
 + shap==0.52.0
 + six==1.17.0
 + slicer==0.0.8
 + threadpoolctl==3.6.0
 + tqdm==4.67.3
 + tzdata==2026.2


In [3]:
import pandas as pd
import numpy as np
import xgboost as xgb
import shap
import matplotlib.pyplot as plt
import seaborn as sns
import os

artifact_dir = r"C:\Users\loq\.gemini\antigravity\brain\08f08955-7051-4470-bcba-a2b3a9aebbfd"
os.makedirs(artifact_dir, exist_ok=True)

print("Loading data...")
df = pd.read_csv('data/ranking_data_upstox_3y.csv')
print(f"Original size: {df.shape}")

df_sample = df.sample(n=min(10000, len(df)), random_state=42)

exclude_cols = ['DateTime', 'DateTime_Hour', 'Query_ID', 'Ticker',
                'Open', 'High', 'Low', 'Close', 'Volume', 'Next_Hour_Return']
feature_cols = [col for col in df_sample.columns if col not in exclude_cols]

X = df_sample[feature_cols]
y_true = df_sample['Next_Hour_Return'].values

print("Loading model...")
model = xgb.Booster()
model.load_model('models/v8_upstox_3y/xgb_long_model.json')

d_X = xgb.DMatrix(X.values)
preds = model.predict(d_X)

print("Calculating SHAP values...")
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X)

print("Generating SHAP Summary Plot...")
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X, show=False)
plt.tight_layout()
plt.savefig(os.path.join(artifact_dir, "shap_summary_plot.png"), dpi=300, bbox_inches='tight')
plt.close('all')

print("Generating SHAP Dependence Plot...")
mean_shap = np.abs(shap_values).mean(axis=0)
top_feature_idx = np.argmax(mean_shap)
top_feature_name = feature_cols[top_feature_idx]

plt.figure(figsize=(10, 6))
shap.dependence_plot(top_feature_name, shap_values, X, show=False)
plt.tight_layout()
plt.savefig(os.path.join(artifact_dir, "shap_dependence_plot.png"), dpi=300, bbox_inches='tight')
plt.close('all')

print("Generating Prediction Bucket Analysis...")
df_bucket = pd.DataFrame({'Score': preds, 'Return': y_true})
df_bucket['Bucket'] = pd.qcut(df_bucket['Score'], q=10, labels=False, duplicates='drop')
bucket_means = df_bucket.groupby('Bucket')['Return'].mean()

plt.figure(figsize=(10, 6))
sns.barplot(x=bucket_means.index, y=bucket_means.values, color='steelblue')
plt.title("Prediction Bucket Analysis (Average Return per Decile)")
plt.xlabel("Prediction Decile (0=Lowest, 9=Highest)")
plt.ylabel("Average Next_Hour_Return")
plt.tight_layout()
plt.savefig(os.path.join(artifact_dir, "prediction_bucket_analysis.png"), dpi=300, bbox_inches='tight')
plt.close('all')

print("Generating Cumulative Return Curve...")
df_bucket['Top5'] = df_bucket['Score'] >= df_bucket['Score'].quantile(0.95)
if 'DateTime' in df_sample.columns:
    df_sorted = df_sample.sort_values('DateTime').copy()
    df_sorted['Score'] = model.predict(xgb.DMatrix(df_sorted[feature_cols].values))
    threshold = df_sorted['Score'].quantile(0.95)
    df_sorted['Strategy_Return'] = np.where(df_sorted['Score'] >= threshold, df_sorted['Next_Hour_Return'], 0)
    df_sorted['Cumulative_Strategy'] = df_sorted['Strategy_Return'].cumsum()
    
    plt.figure(figsize=(12, 6))
    plt.plot(df_sorted['Cumulative_Strategy'].values, label='Strategy (Top 5%)', color='green')
    plt.title("Simulated Cumulative Return Curve (10k Sample)")
    plt.xlabel("Time (Sampled Steps)")
    plt.ylabel("Cumulative Return")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(artifact_dir, "cumulative_return_curve.png"), dpi=300, bbox_inches='tight')
    plt.close('all')

print("All plots generated successfully.")


Loading data...


Original size: (1107513, 96)
Loading model...
Calculating SHAP values...
Generating SHAP Summary Plot...


Generating SHAP Dependence Plot...


Generating Prediction Bucket Analysis...


Generating Cumulative Return Curve...


All plots generated successfully.


In [4]:
# Investigate Bucket 3 vs Bucket 9
print("Re-evaluating Bucket 3 vs Bucket 9")
df_analysis = df_sample.copy()
df_analysis['Score'] = preds
df_analysis['Bucket'] = pd.qcut(df_analysis['Score'], q=10, labels=False, duplicates='drop')

bucket_3 = df_analysis[df_analysis['Bucket'] == 3]
bucket_9 = df_analysis[df_analysis['Bucket'] == 9]

print(f"Bucket 3 size: {len(bucket_3)}")
print(f"Bucket 9 size: {len(bucket_9)}")

print("\n--- Next_Hour_Return Stats ---")
print("Bucket 3 Return Stats:")
print(bucket_3['Next_Hour_Return'].describe())
print("\nBucket 9 Return Stats:")
print(bucket_9['Next_Hour_Return'].describe())

print("\n--- Massive Outliers (> 5% return) ---")
print(f"Bucket 3 outliers: {len(bucket_3[bucket_3['Next_Hour_Return'] > 0.05])}")
print(f"Bucket 9 outliers: {len(bucket_9[bucket_9['Next_Hour_Return'] > 0.05])}")

print("\n--- Feature Differences (Mean) ---")
mean_b3 = bucket_3[feature_cols].mean()
mean_b9 = bucket_9[feature_cols].mean()
diff = (mean_b3 - mean_b9)
print(diff.abs().sort_values(ascending=False).head(10))

print("\n--- Hour Distribution ---")
if 'DateTime_Hour' in df_analysis.columns:
    print("Bucket 3 Hours:\n", bucket_3['DateTime_Hour'].value_counts(normalize=True).head())
    print("Bucket 9 Hours:\n", bucket_9['DateTime_Hour'].value_counts(normalize=True).head())
elif 'Hour' in feature_cols:
    print("Bucket 3 Hours:\n", bucket_3['Hour'].value_counts(normalize=True).head())
    print("Bucket 9 Hours:\n", bucket_9['Hour'].value_counts(normalize=True).head())

if 'Ticker' in df_analysis.columns:
    print("\n--- Top Tickers in Bucket 3 ---")
    print(bucket_3['Ticker'].value_counts().head(5))

Re-evaluating Bucket 3 vs Bucket 9
Bucket 3 size: 1000
Bucket 9 size: 1000

--- Next_Hour_Return Stats ---
Bucket 3 Return Stats:
count    1000.000000
mean        0.001152
std         0.032008
min        -0.046857
25%        -0.002778
50%        -0.000147
75%         0.002952
max         0.995662
Name: Next_Hour_Return, dtype: float64

Bucket 9 Return Stats:
count    1000.000000
mean        0.000638
std         0.010461
min        -0.082958
25%        -0.002778
50%         0.000468
75%         0.003845
max         0.074867
Name: Next_Hour_Return, dtype: float64

--- Massive Outliers (> 5% return) ---
Bucket 3 outliers: 1
Bucket 9 outliers: 2

--- Feature Differences (Mean) ---
IBS              1.692983
Hour             1.468000
Time_To_Close    1.468000
OC_Range         1.280555
Log_Return       1.264932
Return           1.260218
Lower_Shadow     1.156008
Dist_HMA_12      0.981772
IBS_3            0.974133
VWAP_Dist        0.924464
dtype: float64

--- Hour Distribution ---
Bucket 3 Hou

In [5]:
# Investigate Bucket 0
print("Re-evaluating Bucket 0 vs Bucket 9")
bucket_0 = df_analysis[df_analysis['Bucket'] == 0]

print(f"Bucket 0 size: {len(bucket_0)}")

print("\n--- Next_Hour_Return Stats ---")
print("Bucket 0 Return Stats:")
print(bucket_0['Next_Hour_Return'].describe())

print("\n--- Massive Outliers (> 5% or < -5% return) ---")
print(f"Bucket 0 positive outliers (> 5%): {len(bucket_0[bucket_0['Next_Hour_Return'] > 0.05])}")
print(f"Bucket 0 negative outliers (< -5%): {len(bucket_0[bucket_0['Next_Hour_Return'] < -0.05])}")

print("\n--- Feature Differences (Mean) B0 vs B9 ---")
mean_b0 = bucket_0[feature_cols].mean()
diff_0_9 = (mean_b0 - mean_b9)
print(diff_0_9.abs().sort_values(ascending=False).head(10))

print("\nActual differences for those top features (B0 - B9):")
print(diff_0_9[diff_0_9.abs().sort_values(ascending=False).head(10).index])

print("\n--- Hour Distribution ---")
if 'DateTime_Hour' in df_analysis.columns:
    print("Bucket 0 Hours:\n", bucket_0['DateTime_Hour'].value_counts(normalize=True).head())
elif 'Hour' in feature_cols:
    print("Bucket 0 Hours:\n", bucket_0['Hour'].value_counts(normalize=True).head())


Re-evaluating Bucket 0 vs Bucket 9
Bucket 0 size: 1000

--- Next_Hour_Return Stats ---
Bucket 0 Return Stats:
count    1000.000000
mean       -0.000629
std         0.011991
min        -0.068942
25%        -0.005831
50%        -0.000446
75%         0.004236
max         0.088521
Name: Next_Hour_Return, dtype: float64

--- Massive Outliers (> 5% or < -5% return) ---
Bucket 0 positive outliers (> 5%): 3
Bucket 0 negative outliers (< -5%): 2

--- Feature Differences (Mean) B0 vs B9 ---
IBS              2.192213
OC_Range         1.698981
Log_Return       1.698503
Return           1.695444
Hour             1.460000
Time_To_Close    1.460000
IBS_3            1.451527
Buy_Pressure     1.396881
VWAP_Dist        1.375318
Dist_SMA_6       1.311616
dtype: float64

Actual differences for those top features (B0 - B9):
IBS              2.192213
OC_Range         1.698981
Log_Return       1.698503
Return           1.695444
Hour             1.460000
Time_To_Close   -1.460000
IBS_3            1.451527
Buy

In [6]:
print("Running Probability Calibration Analysis...")

df_calib = pd.DataFrame({'Score': preds, 'Return': y_true})
df_calib['Win'] = df_calib['Return'] > 0

score_min = df_calib['Score'].min()
score_max = df_calib['Score'].max()
print(f"Score Range: {score_min:.4f} to {score_max:.4f}")

df_calib['Score_Bin'] = pd.cut(df_calib['Score'], bins=20)
calib_stats = df_calib.groupby('Score_Bin', observed=True).agg(
    Win_Rate=('Win', 'mean'),
    Trade_Count=('Win', 'size'),
    Avg_Return=('Return', 'mean')
).reset_index()

plt.figure(figsize=(12, 6))

ax1 = plt.gca()
ax1.plot([str(round(b.right, 4)) for b in calib_stats['Score_Bin']], calib_stats['Win_Rate'], marker='o', color='blue', label='Win Rate')
ax1.axhline(0.5, color='gray', linestyle='--')
ax1.axhline(0.6, color='green', linestyle='--', label='60% Target')
ax1.set_ylabel('Win Rate', color='blue')
ax1.set_xlabel('Raw Score Threshold (Upper Bound of Bin)')
ax1.tick_params(axis='y', labelcolor='blue')
plt.xticks(rotation=45, ha='right')

ax2 = ax1.twinx()
ax2.bar([str(round(b.right, 4)) for b in calib_stats['Score_Bin']], calib_stats['Trade_Count'], alpha=0.3, color='gray', label='Trade Count')
ax2.set_ylabel('Number of Trades', color='gray')
ax2.tick_params(axis='y', labelcolor='gray')

plt.title("Probability Calibration: Win Rate vs Raw Prediction Score")
plt.tight_layout()
plt.savefig(os.path.join(artifact_dir, "calibration_curve.png"), dpi=300, bbox_inches='tight')
plt.close('all')

for target in [0.55, 0.60]:
    high_win_bins = calib_stats[calib_stats['Win_Rate'] >= target]
    if not high_win_bins.empty:
        best_bin = high_win_bins.iloc[0]
        print(f"\nTarget {target*100}% achieved at Score > {best_bin['Score_Bin'].left:.4f}")
        print(f"Win Rate: {best_bin['Win_Rate']:.2%}, Avg Return: {best_bin['Avg_Return']:.4f}, Sample Size: {best_bin['Trade_Count']}")
    else:
        print(f"\nCould not find a single bin with >= {target*100}% win rate in this sample.")

print("\nDetailed Calibration Stats (Top 5 highest score bins):")
print(calib_stats.tail())

Running Probability Calibration Analysis...
Score Range: -0.1833 to 0.1094



Target 55.00000000000001% achieved at Score > 0.0362
Win Rate: 57.14%, Avg Return: 0.0006, Sample Size: 252

Target 60.0% achieved at Score > 0.0801
Win Rate: 70.00%, Avg Return: -0.0058, Sample Size: 10

Detailed Calibration Stats (Top 5 highest score bins):
           Score_Bin  Win_Rate  Trade_Count  Avg_Return
15  (0.0362, 0.0508]  0.571429          252    0.000583
16  (0.0508, 0.0655]  0.515464           97    0.000279
17  (0.0655, 0.0801]  0.552632           38    0.003170
18  (0.0801, 0.0947]  0.700000           10   -0.005848
19   (0.0947, 0.109]  0.833333            6    0.012764


In [7]:
# Calculate aggregate metrics for Score > 0.0362
threshold = 0.0362
high_score_trades = df_calib[df_calib['Score'] > threshold]

win_rate = high_score_trades['Win'].mean()
avg_return = high_score_trades['Return'].mean()
trade_count = len(high_score_trades)
cum_return = high_score_trades['Return'].sum()

print(f"--- Aggregate Stats for Score > {threshold} ---")
print(f"Total Trades: {trade_count}")
print(f"Aggregate Win Rate: {win_rate:.2%}")
print(f"Average Return per Trade: {avg_return:.4%}")
print(f"Total Cumulative Return: {cum_return:.4%}")


--- Aggregate Stats for Score > 0.0362 ---
Total Trades: 403
Aggregate Win Rate: 56.33%
Average Return per Trade: 0.0776%
Total Cumulative Return: 31.2685%


In [8]:
if 'DateTime' in df_sample.columns:
    date_min = df_sample['DateTime'].min()
    date_max = df_sample['DateTime'].max()
    date_min_dt = pd.to_datetime(date_min)
    date_max_dt = pd.to_datetime(date_max)
    days = (date_max_dt - date_min_dt).days
    print(f"Sample Date Range: {date_min} to {date_max}")
    print(f"Total Timespan: {days} days")
else:
    print("DateTime column not available.")

Sample Date Range: 2022-01-13 13:30:00+05:30 to 2026-05-27 13:30:00+05:30
Total Timespan: 1595 days


In [9]:
import pandas as pd
import numpy as np
import xgboost as xgb
import matplotlib.pyplot as plt
import os

print("Loading data for Realistic Out-of-Sample Calibration...")
df = pd.read_csv('data/ranking_data_upstox_3y.csv')
if 'DateTime' in df.columns:
    df['DateTime'] = pd.to_datetime(df['DateTime'])
    df = df.sort_values('DateTime')
    
    latest_date = df['DateTime'].max()
    oos_start = latest_date - pd.DateOffset(months=12)
    df_test = df[df['DateTime'] >= oos_start].copy()
    print(f"Out-of-Sample period: {oos_start} to {latest_date} ({len(df_test)} rows)")
else:
    df_test = df.tail(int(len(df)*0.2)).copy()

FEE = 0.0010 

exclude_cols = ['DateTime', 'DateTime_Hour', 'Query_ID', 'Ticker',
                'Open', 'High', 'Low', 'Close', 'Volume', 'Next_Hour_Return']
feature_cols = [col for col in df_test.columns if col not in exclude_cols]

X_test = df_test[feature_cols]
d_X = xgb.DMatrix(X_test.values)

print("Loading models...")
long_model = xgb.Booster()
long_model.load_model('models/v8_upstox_3y/xgb_long_model.json')
short_model = xgb.Booster()
short_model.load_model('models/v8_upstox_3y/xgb_short_model.json')

preds_long = long_model.predict(d_X)
preds_short = short_model.predict(d_X)

df_test['Score_Long'] = preds_long
df_test['Score_Short'] = preds_short

df_test['Pnl_Long'] = df_test['Next_Hour_Return'] - FEE
df_test['Win_Long'] = df_test['Pnl_Long'] > 0

df_test['Pnl_Short'] = -df_test['Next_Hour_Return'] - FEE
df_test['Win_Short'] = df_test['Pnl_Short'] > 0

def get_calibration(scores, wins, pnl, bins=20):
    df_c = pd.DataFrame({'Score': scores, 'Win': wins, 'Pnl': pnl})
    df_c['Score_Bin'] = pd.cut(df_c['Score'], bins=bins)
    stats = df_c.groupby('Score_Bin', observed=True).agg(
        Win_Rate=('Win', 'mean'),
        Trade_Count=('Win', 'size'),
        Avg_Pnl=('Pnl', 'mean')
    ).reset_index()
    return stats

calib_long = get_calibration(df_test['Score_Long'], df_test['Win_Long'], df_test['Pnl_Long'])
calib_short = get_calibration(df_test['Score_Short'], df_test['Win_Short'], df_test['Pnl_Short'])

def find_threshold(stats, target=0.50):
    high_win = stats[stats['Win_Rate'] >= target]
    if not high_win.empty:
        best = high_win.iloc[0]
        return best['Score_Bin'].left, best['Win_Rate'], best['Trade_Count'], best['Avg_Pnl']
    return None, None, None, None

print("\n--- LONG MODEL CALIBRATION (Real OOS + 10bps Fees) ---")
for t in [0.50, 0.52]:
    thresh, wr, count, pnl = find_threshold(calib_long, t)
    if thresh is not None:
        print(f"Target {t*100}% -> Score > {thresh:.4f} | Trades: {count} | Avg PnL: {pnl:.4%}")
    else:
        print(f"Target {t*100}% -> Not achievable in this sample.")

print("\n--- SHORT MODEL CALIBRATION (Real OOS + 10bps Fees) ---")
for t in [0.50, 0.52]:
    thresh, wr, count, pnl = find_threshold(calib_short, t)
    if thresh is not None:
        print(f"Target {t*100}% -> Score > {thresh:.4f} | Trades: {count} | Avg PnL: {pnl:.4%}")
    else:
        print(f"Target {t*100}% -> Not achievable in this sample.")

plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
plt.plot([str(round(b.right, 4)) for b in calib_long['Score_Bin']], calib_long['Win_Rate'], marker='o', color='green')
plt.axhline(0.5, color='gray', linestyle='--')
plt.title("Long Model Calibration (OOS + Fees)")
plt.xticks(rotation=45, ha='right')
plt.ylabel('Win Rate')

plt.subplot(1, 2, 2)
plt.plot([str(round(b.right, 4)) for b in calib_short['Score_Bin']], calib_short['Win_Rate'], marker='o', color='red')
plt.axhline(0.5, color='gray', linestyle='--')
plt.title("Short Model Calibration (OOS + Fees)")
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
artifact_dir = r"C:\Users\loq\.gemini\antigravity\brain\08f08955-7051-4470-bcba-a2b3a9aebbfd"
plt.savefig(os.path.join(artifact_dir, "oos_calibration.png"), dpi=300, bbox_inches='tight')
plt.close('all')

print("\nPlots generated successfully.")

Loading data for Realistic Out-of-Sample Calibration...


Out-of-Sample period: 2025-05-27 13:30:00+05:30 to 2026-05-27 13:30:00+05:30 (255206 rows)


Loading models...

--- LONG MODEL CALIBRATION (Real OOS + 10bps Fees) ---
Target 50.0% -> Score > 0.0609 | Trades: 1429 | Avg PnL: 0.0325%
Target 52.0% -> Score > 0.0761 | Trades: 551 | Avg PnL: 0.1795%

--- SHORT MODEL CALIBRATION (Real OOS + 10bps Fees) ---


Target 50.0% -> Score > 0.0624 | Trades: 2294 | Avg PnL: 0.0340%
Target 52.0% -> Score > 0.0624 | Trades: 2294 | Avg PnL: 0.0340%



Plots generated successfully.


In [10]:
FEE_7 = 0.0007

df_test['Pnl_Long_7'] = df_test['Next_Hour_Return'] - FEE_7
df_test['Win_Long_7'] = df_test['Pnl_Long_7'] > 0

df_test['Pnl_Short_7'] = -df_test['Next_Hour_Return'] - FEE_7
df_test['Win_Short_7'] = df_test['Pnl_Short_7'] > 0

calib_long_7 = get_calibration(df_test['Score_Long'], df_test['Win_Long_7'], df_test['Pnl_Long_7'])
calib_short_7 = get_calibration(df_test['Score_Short'], df_test['Win_Short_7'], df_test['Pnl_Short_7'])

print("\n--- LONG MODEL CALIBRATION (Real OOS + 7bps Fees) ---")
for t in [0.50, 0.52, 0.55]:
    thresh, wr, count, pnl = find_threshold(calib_long_7, t)
    if thresh is not None:
        print(f"Target {t*100}% -> Score > {thresh:.4f} | Trades: {count} | Avg PnL: {pnl:.4%}")
    else:
        print(f"Target {t*100}% -> Not achievable.")

print("\n--- SHORT MODEL CALIBRATION (Real OOS + 7bps Fees) ---")
for t in [0.50, 0.52, 0.55]:
    thresh, wr, count, pnl = find_threshold(calib_short_7, t)
    if thresh is not None:
        print(f"Target {t*100}% -> Score > {thresh:.4f} | Trades: {count} | Avg PnL: {pnl:.4%}")
    else:
        print(f"Target {t*100}% -> Not achievable.")


--- LONG MODEL CALIBRATION (Real OOS + 7bps Fees) ---
Target 50.0% -> Score > 0.0609 | Trades: 1429 | Avg PnL: 0.0625%
Target 52.0% -> Score > 0.0761 | Trades: 551 | Avg PnL: 0.2095%
Target 55.00000000000001% -> Score > 0.0761 | Trades: 551 | Avg PnL: 0.2095%

--- SHORT MODEL CALIBRATION (Real OOS + 7bps Fees) ---
Target 50.0% -> Score > 0.0374 | Trades: 10072 | Avg PnL: 0.0023%
Target 52.0% -> Score > 0.0624 | Trades: 2294 | Avg PnL: 0.0640%
Target 55.00000000000001% -> Score > 0.0624 | Trades: 2294 | Avg PnL: 0.0640%


In [11]:
print("\n--- INVERSION STRATEGY ANALYSIS (OOS + 10bps Fees) ---")

df_test['Pnl_Invert_Long'] = -df_test['Next_Hour_Return'] - FEE
df_test['Win_Invert_Long'] = df_test['Pnl_Invert_Long'] > 0

df_test['Pnl_Invert_Short'] = df_test['Next_Hour_Return'] - FEE
df_test['Win_Invert_Short'] = df_test['Pnl_Invert_Short'] > 0

calib_inv_long = get_calibration(df_test['Score_Long'], df_test['Win_Invert_Long'], df_test['Pnl_Invert_Long'])
calib_inv_short = get_calibration(df_test['Score_Short'], df_test['Win_Invert_Short'], df_test['Pnl_Invert_Short'])

print("\nShorting what the Long Model hates (Lowest Long Scores):")
print(calib_inv_long.head(5)[['Score_Bin', 'Win_Rate', 'Trade_Count', 'Avg_Pnl']])

print("\nLonging what the Short Model hates (Lowest Short Scores):")
print(calib_inv_short.head(5)[['Score_Bin', 'Win_Rate', 'Trade_Count', 'Avg_Pnl']])

def find_negative_threshold(stats, target=0.50):
    high_win = stats[(stats['Win_Rate'] >= target) & (stats.index < len(stats)/2)]
    if not high_win.empty:
        best = high_win.iloc[-1] 
        return best['Score_Bin'].right, best['Win_Rate'], best['Trade_Count'], best['Avg_Pnl']
    return None, None, None, None

thresh_inv_long, wr_il, count_il, pnl_il = find_negative_threshold(calib_inv_long, 0.50)
if thresh_inv_long is not None:
    print(f"\nTarget 50% Win Rate for Inverting Long Model achieved at Score < {thresh_inv_long:.4f} | Trades: {count_il} | Avg PnL: {pnl_il:.4%}")

thresh_inv_short, wr_is, count_is, pnl_is = find_negative_threshold(calib_inv_short, 0.50)
if thresh_inv_short is not None:
    print(f"Target 50% Win Rate for Inverting Short Model achieved at Score < {thresh_inv_short:.4f} | Trades: {count_is} | Avg PnL: {pnl_is:.4%}")


--- INVERSION STRATEGY ANALYSIS (OOS + 10bps Fees) ---

Shorting what the Long Model hates (Lowest Long Scores):
          Score_Bin  Win_Rate  Trade_Count   Avg_Pnl
0  (-0.183, -0.167]  0.614914          818  0.003742
1  (-0.167, -0.152]  0.525681         3894  0.000390
2  (-0.152, -0.137]  0.486554         7586 -0.000406
3  (-0.137, -0.121]  0.468142         7141 -0.000553
4  (-0.121, -0.106]  0.461187         7433 -0.000643

Longing what the Short Model hates (Lowest Short Scores):
          Score_Bin  Win_Rate  Trade_Count   Avg_Pnl
0  (-0.263, -0.238]  0.488024         3340  0.000056
1  (-0.238, -0.213]  0.452576         3416 -0.000191
2  (-0.213, -0.188]  0.470524         3647 -0.000133
3  (-0.188, -0.163]  0.439881         6745 -0.000450
4  (-0.163, -0.138]  0.427430         8764 -0.000610

Target 50% Win Rate for Inverting Long Model achieved at Score < -0.1520 | Trades: 3894 | Avg PnL: 0.0390%


In [12]:
print("Running Quarterly Consistency Analysis on OOS Data (May 2025 - May 2026)...\n")

df_test['Quarter'] = df_test['DateTime'].dt.to_period('Q')

long_trades = df_test[df_test['Score_Long'] > 0.0761]
long_quarterly = long_trades.groupby('Quarter').agg(
    Trades=('Win_Long', 'size'),
    Win_Rate=('Win_Long', 'mean'),
    Avg_PnL=('Pnl_Long', 'mean')
).reset_index()

print("--- LONG STRATEGY CONSISTENCY (Score > 0.0761) ---")
print(long_quarterly.to_string(index=False, formatters={
    'Win_Rate': '{:.2%}'.format,
    'Avg_PnL': '{:.4%}'.format
}))

short_trades = df_test[df_test['Score_Long'] < -0.1520]
short_quarterly = short_trades.groupby('Quarter').agg(
    Trades=('Win_Invert_Long', 'size'),
    Win_Rate=('Win_Invert_Long', 'mean'),
    Avg_PnL=('Pnl_Invert_Long', 'mean')
).reset_index()

print("\n--- INVERTED SHORT STRATEGY CONSISTENCY (Score_Long < -0.1520) ---")
print(short_quarterly.to_string(index=False, formatters={
    'Win_Rate': '{:.2%}'.format,
    'Avg_PnL': '{:.4%}'.format
}))

import matplotlib.pyplot as plt
import os

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

long_quarterly['Quarter_Str'] = long_quarterly['Quarter'].astype(str)
ax1.bar(long_quarterly['Quarter_Str'], long_quarterly['Win_Rate'], color='green', alpha=0.7)
ax1.axhline(0.5, color='gray', linestyle='--')
ax1.set_title("Long Strategy Win Rate by Quarter")
ax1.set_ylabel("Win Rate")
ax1.set_ylim(0, 1.0)
for i, v in enumerate(long_quarterly['Win_Rate']):
    ax1.text(i, v + 0.02, f"{v:.1%}", ha='center', va='bottom')

short_quarterly['Quarter_Str'] = short_quarterly['Quarter'].astype(str)
ax2.bar(short_quarterly['Quarter_Str'], short_quarterly['Win_Rate'], color='red', alpha=0.7)
ax2.axhline(0.5, color='gray', linestyle='--')
ax2.set_title("Inverted Short Win Rate by Quarter")
ax2.set_ylabel("Win Rate")
ax2.set_ylim(0, 1.0)
for i, v in enumerate(short_quarterly['Win_Rate']):
    ax2.text(i, v + 0.02, f"{v:.1%}", ha='center', va='bottom')

artifact_dir = r"C:\Users\loq\.gemini\antigravity\brain\08f08955-7051-4470-bcba-a2b3a9aebbfd"
plt.savefig(os.path.join(artifact_dir, "quarterly_consistency.png"), dpi=300, bbox_inches='tight')
plt.close()


Running Quarterly Consistency Analysis on OOS Data (May 2025 - May 2026)...

--- LONG STRATEGY CONSISTENCY (Score > 0.0761) ---
Quarter  Trades Win_Rate  Avg_PnL
 2025Q2      61   60.66%  0.3106%
 2025Q3     167   55.69%  0.1907%
 2025Q4     168   57.14%  0.1747%
 2026Q1     174   51.15% -0.2122%
 2026Q2     131   64.12%  0.6749%

--- INVERTED SHORT STRATEGY CONSISTENCY (Score_Long < -0.1520) ---
Quarter  Trades Win_Rate  Avg_PnL
 2025Q2     423   52.48%  0.0197%
 2025Q3    1140   53.33%  0.0307%
 2025Q4    1110   55.86%  0.0954%
 2026Q1    1246   58.03%  0.3632%
 2026Q2     721   47.30% -0.1925%


C:\Users\loq\AppData\Local\Temp\ipykernel_38972\3237097229.py:3: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df_test['Quarter'] = df_test['DateTime'].dt.to_period('Q')


In [13]:
print("\n--- CONVICTION SCORE vs TIME OF DAY ANALYSIS ---")

if 'DateTime_Hour' in df_test.columns:
    df_test['Hour_Of_Day'] = df_test['DateTime_Hour'].dt.hour
elif 'DateTime' in df_test.columns:
    df_test['Hour_Of_Day'] = df_test['DateTime'].dt.hour
else:
    df_test['Hour_Of_Day'] = df_test['Hour'] 

df_test['Score_Bin_Simple'] = pd.qcut(df_test['Score_Long'], q=5, labels=['Lowest', 'Low', 'Medium', 'High', 'Highest'], duplicates='drop')

time_score_stats = df_test.groupby(['Hour_Of_Day', 'Score_Bin_Simple'], observed=True).agg(
    Win_Rate=('Win_Long', 'mean'),
    Avg_PnL=('Pnl_Long', 'mean'),
    Trades=('Win_Long', 'size')
).reset_index()

import seaborn as sns
plt.figure(figsize=(10, 6))
heatmap_data = time_score_stats.pivot(index="Hour_Of_Day", columns="Score_Bin_Simple", values="Avg_PnL")
sns.heatmap(heatmap_data, annot=True, fmt=".2%", cmap="RdYlGn", center=0)
plt.title("Average PnL by Time of Day and Score Quintile (10bps fees)")
plt.ylabel("Hour of Day (9=9AM, 14=2PM)")
plt.xlabel("Conviction Score Quintile")
artifact_dir = r"C:\Users\loq\.gemini\antigravity\brain\08f08955-7051-4470-bcba-a2b3a9aebbfd"
plt.savefig(os.path.join(artifact_dir, "time_of_day_conviction.png"), dpi=300, bbox_inches='tight')
plt.close()

print("Time of Day heatmap generated.")

high_conviction = df_test[df_test['Score_Long'] > 0.076]
high_conv_time = high_conviction.groupby('Hour_Of_Day').agg(
    Trades=('Win_Long', 'size'),
    Win_Rate=('Win_Long', 'mean'),
    Avg_PnL=('Pnl_Long', 'mean')
).reset_index()

print("\nHigh Conviction (>0.076) Performance by Hour:")
print(high_conv_time.to_string(index=False, formatters={
    'Win_Rate': '{:.2%}'.format,
    'Avg_PnL': '{:.4%}'.format
}))


--- CONVICTION SCORE vs TIME OF DAY ANALYSIS ---


AttributeError: Can only use .dt accessor with datetimelike values

In [14]:
print("\n--- CONVICTION SCORE vs TIME OF DAY ANALYSIS ---")

df_test['Hour_Of_Day'] = pd.to_datetime(df_test['DateTime']).dt.hour

df_test['Score_Bin_Simple'] = pd.qcut(df_test['Score_Long'], q=5, labels=['Lowest', 'Low', 'Medium', 'High', 'Highest'], duplicates='drop')

time_score_stats = df_test.groupby(['Hour_Of_Day', 'Score_Bin_Simple'], observed=True).agg(
    Win_Rate=('Win_Long', 'mean'),
    Avg_PnL=('Pnl_Long', 'mean'),
    Trades=('Win_Long', 'size')
).reset_index()

import seaborn as sns
import matplotlib.pyplot as plt
import os

plt.figure(figsize=(10, 6))
heatmap_data = time_score_stats.pivot(index="Hour_Of_Day", columns="Score_Bin_Simple", values="Avg_PnL")
sns.heatmap(heatmap_data, annot=True, fmt=".2%", cmap="RdYlGn", center=0)
plt.title("Average PnL by Time of Day and Score Quintile (10bps fees)")
plt.ylabel("Hour of Day (9=9AM, 14=2PM)")
plt.xlabel("Conviction Score Quintile")
artifact_dir = r"C:\Users\loq\.gemini\antigravity\brain\08f08955-7051-4470-bcba-a2b3a9aebbfd"
plt.savefig(os.path.join(artifact_dir, "time_of_day_conviction.png"), dpi=300, bbox_inches='tight')
plt.close()

print("Time of Day heatmap generated.")

high_conviction = df_test[df_test['Score_Long'] > 0.076]
high_conv_time = high_conviction.groupby('Hour_Of_Day').agg(
    Trades=('Win_Long', 'size'),
    Win_Rate=('Win_Long', 'mean'),
    Avg_PnL=('Pnl_Long', 'mean')
).reset_index()

print("\nHigh Conviction (>0.076) Performance by Hour:")
print(high_conv_time.to_string(index=False, formatters={
    'Win_Rate': '{:.2%}'.format,
    'Avg_PnL': '{:.4%}'.format
}))


--- CONVICTION SCORE vs TIME OF DAY ANALYSIS ---


Time of Day heatmap generated.

High Conviction (>0.076) Performance by Hour:
 Hour_Of_Day  Trades Win_Rate Avg_PnL
          13      95   54.74% 0.0836%
          14     610   57.38% 0.2026%


In [15]:
print("\n--- INVERTED SHORT CONVICTION vs TIME OF DAY ANALYSIS ---")

high_conv_short = df_test[df_test['Score_Long'] < -0.1520]

short_conv_time = high_conv_short.groupby('Hour_Of_Day').agg(
    Trades=('Win_Invert_Long', 'size'),
    Win_Rate=('Win_Invert_Long', 'mean'),
    Avg_PnL=('Pnl_Invert_Long', 'mean')
).reset_index()

print("High Conviction Inverted Short (< -0.152) Performance by Hour:")
print(short_conv_time.to_string(index=False, formatters={
    'Win_Rate': '{:.2%}'.format,
    'Avg_PnL': '{:.4%}'.format
}))

sniper_short = df_test[df_test['Score_Long'] < -0.1670]
sniper_time = sniper_short.groupby('Hour_Of_Day').agg(
    Trades=('Win_Invert_Long', 'size'),
    Win_Rate=('Win_Invert_Long', 'mean'),
    Avg_PnL=('Pnl_Invert_Long', 'mean')
).reset_index()

print("\nSniper Inverted Short (< -0.167) Performance by Hour:")
print(sniper_time.to_string(index=False, formatters={
    'Win_Rate': '{:.2%}'.format,
    'Avg_PnL': '{:.4%}'.format
}))


--- INVERTED SHORT CONVICTION vs TIME OF DAY ANALYSIS ---
High Conviction Inverted Short (< -0.152) Performance by Hour:
 Hour_Of_Day  Trades Win_Rate Avg_PnL
          13     675   49.19% 0.0253%
          14    3965   55.03% 0.1125%

Sniper Inverted Short (< -0.167) Performance by Hour:
 Hour_Of_Day  Trades Win_Rate Avg_PnL
          14     818   61.49% 0.3742%


In [16]:
print("\n--- TIMESTAMP VERIFICATION ---")
df_14 = df_test[df_test['Hour_Of_Day'] == 14]
unique_times_14 = df_14['DateTime'].dt.time.unique()
print(f"Unique execution times for trades marked as Hour 14: {unique_times_14}")

all_times = df_test['DateTime'].dt.time.unique()
print(f"All unique execution times available in the dataset: {all_times}")


--- TIMESTAMP VERIFICATION ---
Unique execution times for trades marked as Hour 14: [datetime.time(14, 30)]
All unique execution times available in the dataset: [datetime.time(13, 30) datetime.time(14, 30) datetime.time(9, 30)
 datetime.time(10, 30) datetime.time(11, 30) datetime.time(12, 30)]


In [17]:
print("\n--- DATA LEAKAGE VERIFICATION ---")

train_mask = df['Query_ID'].isin(train_qids)
test_mask = df['Query_ID'].isin(test_qids)

df_train = df[train_mask]
df_test = df[test_mask]

train_start = pd.to_datetime(df_train['DateTime']).min()
train_end = pd.to_datetime(df_train['DateTime']).max()

test_start = pd.to_datetime(df_test['DateTime']).min()
test_end = pd.to_datetime(df_test['DateTime']).max()

print(f"TRAINING DATA (80%):")
print(f"  Start Date: {train_start}")
print(f"  End Date:   {train_end}")
print(f"  Total Rows: {len(df_train):,}")

print(f"\nTESTING DATA (20% Out-of-Sample):")
print(f"  Start Date: {test_start}")
print(f"  End Date:   {test_end}")
print(f"  Total Rows: {len(df_test):,}")

overlap = df_train[df_train['DateTime'].isin(df_test['DateTime'])]
print(f"\nData Leakage Check - Overlapping timestamps between Train & Test: {len(overlap)}")


--- DATA LEAKAGE VERIFICATION ---


NameError: name 'train_qids' is not defined

In [18]:
print("\n--- DATA LEAKAGE VERIFICATION ---")

unique_query_ids = np.sort(df['Query_ID'].unique())
split_idx = int(len(unique_query_ids) * 0.8)

train_qids = unique_query_ids[:split_idx]
test_qids = unique_query_ids[split_idx:]

train_mask = df['Query_ID'].isin(train_qids)
test_mask = df['Query_ID'].isin(test_qids)

df_train = df[train_mask]
df_test = df[test_mask]

train_start = pd.to_datetime(df_train['DateTime']).min()
train_end = pd.to_datetime(df_train['DateTime']).max()

test_start = pd.to_datetime(df_test['DateTime']).min()
test_end = pd.to_datetime(df_test['DateTime']).max()

print(f"TRAINING DATA (80%):")
print(f"  Start Date: {train_start}")
print(f"  End Date:   {train_end}")
print(f"  Total Rows: {len(df_train):,}")

print(f"\nTESTING DATA (20% Out-of-Sample):")
print(f"  Start Date: {test_start}")
print(f"  End Date:   {test_end}")
print(f"  Total Rows: {len(df_test):,}")

overlap = df_train[df_train['DateTime'].isin(df_test['DateTime'])]
print(f"\nData Leakage Check - Overlapping timestamps between Train & Test: {len(overlap)}")


--- DATA LEAKAGE VERIFICATION ---


TRAINING DATA (80%):
  Start Date: 2022-01-13 10:30:00+05:30
  End Date:   2025-07-10 11:30:00+05:30
  Total Rows: 885,159

TESTING DATA (20% Out-of-Sample):
  Start Date: 2025-07-10 12:30:00+05:30
  End Date:   2026-05-27 13:30:00+05:30
  Total Rows: 222,354

Data Leakage Check - Overlapping timestamps between Train & Test: 0


In [19]:
print("\n--- DEEP DIVE: SHORT MODEL EDGE EXPLORATION ---")

print("\nTop 10 Highest Scoring Bins for Short Model (OOS, 10bps Fees):")
top_short_bins = calib_short.tail(10).sort_values('Score_Bin', ascending=False)
print(top_short_bins[['Score_Bin', 'Win_Rate', 'Trade_Count', 'Avg_Pnl']])

print("\nExtreme Threshold Search:")
for t in [0.52, 0.55, 0.58, 0.60]:
    thresh, wr, count, pnl = find_threshold(calib_short, t)
    if thresh is not None:
        print(f"Target {t*100}% -> Score > {thresh:.4f} | Trades: {count} | Avg PnL: {pnl:.4%}")
    else:
        print(f"Target {t*100}% -> Not achievable.")

best_thresh = 0.07  # Arbitrary starting high threshold to see if there's any volume
if 'Hour_Of_Day' in df_test.columns:
    print(f"\nTime of Day Analysis for Short Model (Score > {best_thresh}):")
    best_shorts = df_test[df_test['Score_Short'] > best_thresh]
    best_shorts_time = best_shorts.groupby('Hour_Of_Day').agg(
        Trades=('Win_Short', 'size'),
        Win_Rate=('Win_Short', 'mean'),
        Avg_PnL=('Pnl_Short', 'mean')
    ).reset_index()
    print(best_shorts_time.to_string(index=False, formatters={
        'Win_Rate': '{:.2%}'.format,
        'Avg_PnL': '{:.4%}'.format
    }))



--- DEEP DIVE: SHORT MODEL EDGE EXPLORATION ---

Top 10 Highest Scoring Bins for Short Model (OOS, 10bps Fees):
            Score_Bin  Win_Rate  Trade_Count   Avg_Pnl
19     (0.212, 0.237]  1.000000            2  0.031959
18     (0.187, 0.212]  0.923077           13  0.020404
17     (0.162, 0.187]  0.821429           28  0.032327
16     (0.137, 0.162]  0.684211           76  0.007184
15     (0.112, 0.137]  0.660194          206  0.004844
14    (0.0874, 0.112]  0.587678          633  0.001998
13   (0.0624, 0.0874]  0.536181         2294  0.000340
12   (0.0374, 0.0624]  0.481632        10072 -0.000277
11   (0.0124, 0.0374]  0.432171        37624 -0.000907
10  (-0.0127, 0.0124]  0.407200        40113 -0.001042

Extreme Threshold Search:
Target 52.0% -> Score > 0.0624 | Trades: 2294 | Avg PnL: 0.0340%
Target 55.00000000000001% -> Score > 0.0874 | Trades: 633 | Avg PnL: 0.1998%
Target 57.99999999999999% -> Score > 0.0874 | Trades: 633 | Avg PnL: 0.1998%
Target 60.0% -> Score > 0.1120 | Tra

In [20]:
best_shorts = df_test[df_test['Score_Short'] > 0.0874]
best_shorts_time = best_shorts.groupby('Hour_Of_Day').agg(
    Trades=('Win_Short', 'size'),
    Win_Rate=('Win_Short', 'mean'),
    Avg_PnL=('Pnl_Short', 'mean')
).reset_index()

print("\nShort Model (Score > 0.0874) Performance by Hour:")
print(best_shorts_time.to_string(index=False, formatters={
    'Win_Rate': '{:.2%}'.format,
    'Avg_PnL': '{:.4%}'.format
}))

KeyError: 'Score_Short'

In [21]:
bst_short = xgb.Booster()
bst_short.load_model('models/v8_upstox_3y/xgb_short_model.json')

feature_cols = [col for col in df.columns if col not in ['DateTime', 'DateTime_Hour', 'Query_ID', 'Ticker', 'Open', 'High', 'Low', 'Close', 'Volume', 'Next_Hour_Return']]
df_test['Score_Short'] = bst_short.predict(xgb.DMatrix(df_test[feature_cols].values))

FEE = 0.0010
df_test['Pnl_Short'] = -df_test['Next_Hour_Return'] - FEE
df_test['Win_Short'] = df_test['Pnl_Short'] > 0

if 'Hour_Of_Day' not in df_test.columns:
    df_test['Hour_Of_Day'] = pd.to_datetime(df_test['DateTime']).dt.hour

best_shorts = df_test[df_test['Score_Short'] > 0.0874]
best_shorts_time = best_shorts.groupby('Hour_Of_Day').agg(
    Trades=('Win_Short', 'size'),
    Win_Rate=('Win_Short', 'mean'),
    Avg_PnL=('Pnl_Short', 'mean')
).reset_index()

print("\nShort Model (Score > 0.0874) Performance by Hour:")
print(best_shorts_time.to_string(index=False, formatters={
    'Win_Rate': '{:.2%}'.format,
    'Avg_PnL': '{:.4%}'.format
}))


Short Model (Score > 0.0874) Performance by Hour:
 Hour_Of_Day  Trades Win_Rate  Avg_PnL
           9      12   33.33% -0.0656%
          10      10   50.00%  0.0566%
          11      15   46.67% -0.2159%
          12      27   33.33% -0.1077%
          13     235   54.47%  0.1357%
          14     564   68.62%  0.6504%


In [22]:
print("\n--- DEEP DIVE: INVERTED SHORT MODEL EXPLORATION (USING SHORT MODEL FOR LONGS) ---")

print("\nTop 10 Lowest Scoring Bins for Short Model (Used for Long Trades):")
top_inv_short_bins = calib_inv_short.head(10).sort_values('Score_Bin', ascending=True)
print(top_inv_short_bins[['Score_Bin', 'Win_Rate', 'Trade_Count', 'Avg_Pnl']])

print("\nExtreme Negative Threshold Search (Target Win Rate for Longs):")
for t in [0.48, 0.50, 0.52]:
    thresh, wr, count, pnl = find_negative_threshold(calib_inv_short, t)
    if thresh is not None:
        print(f"Target {t*100}% -> Score < {thresh:.4f} | Trades: {count} | Avg PnL: {pnl:.4%}")
    else:
        print(f"Target {t*100}% -> Not achievable.")

extreme_inv_short = df_test[df_test['Score_Short'] < -0.238]
if len(extreme_inv_short) > 0:
    print("\nInverted Short Model (Score < -0.238) Performance by Hour:")
    inv_short_time = extreme_inv_short.groupby('Hour_Of_Day').agg(
        Trades=('Win_Invert_Short', 'size'),
        Win_Rate=('Win_Invert_Short', 'mean'),
        Avg_PnL=('Pnl_Invert_Short', 'mean')
    ).reset_index()
    print(inv_short_time.to_string(index=False, formatters={
        'Win_Rate': '{:.2%}'.format,
        'Avg_PnL': '{:.4%}'.format
    }))



--- DEEP DIVE: INVERTED SHORT MODEL EXPLORATION (USING SHORT MODEL FOR LONGS) ---

Top 10 Lowest Scoring Bins for Short Model (Used for Long Trades):
            Score_Bin  Win_Rate  Trade_Count   Avg_Pnl
0    (-0.263, -0.238]  0.488024         3340  0.000056
1    (-0.238, -0.213]  0.452576         3416 -0.000191
2    (-0.213, -0.188]  0.470524         3647 -0.000133
3    (-0.188, -0.163]  0.439881         6745 -0.000450
4    (-0.163, -0.138]  0.427430         8764 -0.000610
5    (-0.138, -0.113]  0.416690         7190 -0.000765
6   (-0.113, -0.0877]  0.379421        24374 -0.000910
7  (-0.0877, -0.0627]  0.375822        37571 -0.000950
8  (-0.0627, -0.0377]  0.378518        36207 -0.000906
9  (-0.0377, -0.0127]  0.378827        32891 -0.000931

Extreme Negative Threshold Search (Target Win Rate for Longs):
Target 48.0% -> Score < -0.2380 | Trades: 3340 | Avg PnL: 0.0056%
Target 50.0% -> Not achievable.
Target 52.0% -> Not achievable.

Inverted Short Model (Score < -0.238) Performance

KeyError: "Label(s) ['Pnl_Invert_Short', 'Win_Invert_Short'] do not exist"

In [23]:
df_test['Pnl_Invert_Short'] = df_test['Next_Hour_Return'] - FEE
df_test['Win_Invert_Short'] = df_test['Pnl_Invert_Short'] > 0

extreme_inv_short = df_test[df_test['Score_Short'] < -0.238]
if len(extreme_inv_short) > 0:
    print("\nInverted Short Model (Score < -0.238) Performance by Hour:")
    inv_short_time = extreme_inv_short.groupby('Hour_Of_Day').agg(
        Trades=('Win_Invert_Short', 'size'),
        Win_Rate=('Win_Invert_Short', 'mean'),
        Avg_PnL=('Pnl_Invert_Short', 'mean')
    ).reset_index()
    print(inv_short_time.to_string(index=False, formatters={
        'Win_Rate': '{:.2%}'.format,
        'Avg_PnL': '{:.4%}'.format
    }))


Inverted Short Model (Score < -0.238) Performance by Hour:
 Hour_Of_Day  Trades Win_Rate  Avg_PnL
          13    1142   42.91% -0.0504%
          14    1691   53.40%  0.0339%


In [24]:
import itertools
import pandas as pd

print("\n--- INITIATING COMBINED THRESHOLD PERMUTATION SWEEP ---")

long_thresholds_high = [0.04, 0.05, 0.06, 0.07, 0.0761, 0.08]
long_thresholds_low = [-0.10, -0.12, -0.14, -0.152, -0.167, -0.18]

short_thresholds_low = [-0.10, -0.15, -0.18, -0.20, -0.238]
short_thresholds_high = [0.06, 0.07, 0.0874, 0.10, 0.112]

print("\n--- COMBINED LONG STRATEGY SWEEP (Score_Long > L_Thresh & Score_Short < S_Thresh) ---")
long_results = []
for l_thresh, s_thresh in itertools.product(long_thresholds_high, short_thresholds_low):
    mask = (df_test['Score_Long'] > l_thresh) & (df_test['Score_Short'] < s_thresh)
    trades = mask.sum()
    if trades >= 50:
        win_rate = df_test.loc[mask, 'Win_Long'].mean()
        avg_pnl = df_test.loc[mask, 'Pnl_Long'].mean()
        long_results.append({
            'L_Thresh (>X)': l_thresh,
            'S_Thresh (<Y)': s_thresh,
            'Trades': trades,
            'Win_Rate': win_rate,
            'Avg_PnL': avg_pnl
        })

long_df = pd.DataFrame(long_results)
if not long_df.empty:
    long_df = long_df.sort_values(by=['Win_Rate', 'Avg_PnL'], ascending=[False, False]).head(15)
    print(long_df.to_string(index=False, formatters={'Win_Rate': '{:.2%}'.format, 'Avg_PnL': '{:.4%}'.format}))
else:
    print("No configurations with >50 trades found.")

print("\n--- COMBINED SHORT STRATEGY SWEEP (Score_Short > S_Thresh & Score_Long < L_Thresh) ---")
short_results = []
for s_thresh, l_thresh in itertools.product(short_thresholds_high, long_thresholds_low):
    mask = (df_test['Score_Short'] > s_thresh) & (df_test['Score_Long'] < l_thresh)
    trades = mask.sum()
    if trades >= 50:
        win_rate = df_test.loc[mask, 'Win_Short'].mean()
        avg_pnl = df_test.loc[mask, 'Pnl_Short'].mean()
        short_results.append({
            'S_Thresh (>X)': s_thresh,
            'L_Thresh (<Y)': l_thresh,
            'Trades': trades,
            'Win_Rate': win_rate,
            'Avg_PnL': avg_pnl
        })

short_df = pd.DataFrame(short_results)
if not short_df.empty:
    short_df = short_df.sort_values(by=['Win_Rate', 'Avg_PnL'], ascending=[False, False]).head(15)
    print(short_df.to_string(index=False, formatters={'Win_Rate': '{:.2%}'.format, 'Avg_PnL': '{:.4%}'.format}))
else:
    print("No configurations with >50 trades found.")


--- INITIATING COMBINED THRESHOLD PERMUTATION SWEEP ---

--- COMBINED LONG STRATEGY SWEEP (Score_Long > L_Thresh & Score_Short < S_Thresh) ---


KeyError: 'Score_Long'

In [25]:
bst_long = xgb.Booster()
bst_long.load_model('models/v8_upstox_3y/xgb_long_model.json')
df_test['Score_Long'] = bst_long.predict(xgb.DMatrix(df_test[feature_cols].values))

df_test['Pnl_Long'] = df_test['Next_Hour_Return'] - FEE
df_test['Win_Long'] = df_test['Pnl_Long'] > 0

long_results = []
for l_thresh, s_thresh in itertools.product(long_thresholds_high, short_thresholds_low):
    mask = (df_test['Score_Long'] > l_thresh) & (df_test['Score_Short'] < s_thresh)
    trades = mask.sum()
    if trades >= 50:
        win_rate = df_test.loc[mask, 'Win_Long'].mean()
        avg_pnl = df_test.loc[mask, 'Pnl_Long'].mean()
        long_results.append({
            'L_Thresh (>X)': l_thresh,
            'S_Thresh (<Y)': s_thresh,
            'Trades': trades,
            'Win_Rate': win_rate,
            'Avg_PnL': avg_pnl
        })

print("\n--- COMBINED LONG STRATEGY SWEEP ---")
long_df = pd.DataFrame(long_results)
if not long_df.empty:
    long_df = long_df.sort_values(by=['Win_Rate', 'Avg_PnL'], ascending=[False, False]).head(10)
    print(long_df.to_string(index=False, formatters={'Win_Rate': '{:.2%}'.format, 'Avg_PnL': '{:.4%}'.format}))

print("\n--- COMBINED SHORT STRATEGY SWEEP ---")
short_results = []
for s_thresh, l_thresh in itertools.product(short_thresholds_high, long_thresholds_low):
    mask = (df_test['Score_Short'] > s_thresh) & (df_test['Score_Long'] < l_thresh)
    trades = mask.sum()
    if trades >= 50:
        win_rate = df_test.loc[mask, 'Win_Short'].mean()
        avg_pnl = df_test.loc[mask, 'Pnl_Short'].mean()
        short_results.append({
            'S_Thresh (>X)': s_thresh,
            'L_Thresh (<Y)': l_thresh,
            'Trades': trades,
            'Win_Rate': win_rate,
            'Avg_PnL': avg_pnl
        })

short_df = pd.DataFrame(short_results)
if not short_df.empty:
    short_df = short_df.sort_values(by=['Win_Rate', 'Avg_PnL'], ascending=[False, False]).head(10)
    print(short_df.to_string(index=False, formatters={'Win_Rate': '{:.2%}'.format, 'Avg_PnL': '{:.4%}'.format}))


--- COMBINED LONG STRATEGY SWEEP ---
 L_Thresh (>X)  S_Thresh (<Y)  Trades Win_Rate Avg_PnL
        0.0800         -0.200     352   59.66% 0.2008%
        0.0761         -0.200     477   58.28% 0.1559%
        0.0800         -0.238     182   58.24% 0.2452%
        0.0800         -0.180     399   57.89% 0.1635%
        0.0761         -0.180     541   57.30% 0.1347%
        0.0800         -0.150     433   57.27% 0.1987%
        0.0800         -0.100     450   57.11% 0.2050%
        0.0761         -0.100     616   56.49% 0.1788%
        0.0761         -0.150     587   56.39% 0.1610%
        0.0761         -0.238     251   56.18% 0.1833%

--- COMBINED SHORT STRATEGY SWEEP ---
 S_Thresh (>X)  L_Thresh (<Y)  Trades Win_Rate Avg_PnL
        0.1120         -0.167     157   75.80% 1.2471%
        0.1000         -0.167     217   74.65% 1.1304%
        0.1120         -0.140     251   72.11% 0.9965%
        0.1120         -0.152     235   71.91% 1.0153%
        0.0874         -0.167     282   70.

C:\Users\loq\AppData\Local\Temp\ipykernel_38972\3229476635.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_test['Pnl_Long'] = df_test['Next_Hour_Return'] - FEE
C:\Users\loq\AppData\Local\Temp\ipykernel_38972\3229476635.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_test['Win_Long'] = df_test['Pnl_Long'] > 0


In [26]:
long_mask = (df_test['Score_Long'] > 0.080) & (df_test['Score_Short'] < -0.200)
short_mask = (df_test['Score_Short'] > 0.0874) & (df_test['Score_Long'] < -0.167)

print("Dual-Confirmed LONG Trades by Hour:")
print(df_test[long_mask].groupby('Hour_Of_Day').size())

print("\nDual-Confirmed SHORT Trades by Hour:")
print(df_test[short_mask].groupby('Hour_Of_Day').size())

Dual-Confirmed LONG Trades by Hour:
Hour_Of_Day
13     37
14    315
dtype: int64

Dual-Confirmed SHORT Trades by Hour:
Hour_Of_Day
14    282
dtype: int64


In [27]:
if 'YearQuarter' not in df_test.columns:
    df_test['YearQuarter'] = pd.to_datetime(df_test['DateTime']).dt.to_period('Q').astype(str)

print("Dual-Confirmed LONG Trades by Quarter:")
long_qtr = df_test[long_mask].groupby('YearQuarter').agg(
    Trades=('Win_Long', 'size'),
    Win_Rate=('Win_Long', 'mean'),
    Avg_PnL=('Pnl_Long', 'mean')
).reset_index()
print(long_qtr.to_string(index=False, formatters={'Win_Rate': '{:.2%}'.format, 'Avg_PnL': '{:.4%}'.format}))

print("\nDual-Confirmed SHORT Trades by Quarter:")
short_qtr = df_test[short_mask].groupby('YearQuarter').agg(
    Trades=('Win_Short', 'size'),
    Win_Rate=('Win_Short', 'mean'),
    Avg_PnL=('Pnl_Short', 'mean')
).reset_index()
print(short_qtr.to_string(index=False, formatters={'Win_Rate': '{:.2%}'.format, 'Avg_PnL': '{:.4%}'.format}))

Dual-Confirmed LONG Trades by Quarter:
YearQuarter  Trades Win_Rate  Avg_PnL
     2025Q3      84   54.76%  0.1231%
     2025Q4      95   61.05%  0.2328%
     2026Q1     100   54.00% -0.2319%
     2026Q2      73   71.23%  0.8416%

Dual-Confirmed SHORT Trades by Quarter:
YearQuarter  Trades Win_Rate  Avg_PnL
     2025Q3      43   65.12%  0.4202%
     2025Q4      64   75.00%  1.3186%
     2026Q1     118   82.20%  1.4075%
     2026Q2      57   45.61% -0.3824%


C:\Users\loq\AppData\Local\Temp\ipykernel_38972\2972701428.py:2: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df_test['YearQuarter'] = pd.to_datetime(df_test['DateTime']).dt.to_period('Q').astype(str)
C:\Users\loq\AppData\Local\Temp\ipykernel_38972\2972701428.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_test['YearQuarter'] = pd.to_datetime(df_test['DateTime']).dt.to_period('Q').astype(str)


In [28]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

print("Generating high-resolution heatmap for Dual-Confirmation sweet spots...")

# 1. LONG STRATEGY HEATMAP
l_grid_long = np.arange(0.04, 0.085, 0.005)
s_grid_long = np.arange(-0.24, -0.09, 0.02)

results_long = []
for l_thresh in l_grid_long:
    for s_thresh in s_grid_long:
        mask = (df_test['Score_Long'] > l_thresh) & (df_test['Score_Short'] < s_thresh)
        trades = mask.sum()
        if trades >= 30:
            wr = df_test.loc[mask, 'Win_Long'].mean()
            pnl = df_test.loc[mask, 'Pnl_Long'].mean()
        else:
            wr = np.nan
            pnl = np.nan
        results_long.append({
            'Score_Long': round(l_thresh, 3),
            'Score_Short': round(s_thresh, 3),
            'Win_Rate': wr,
            'PnL': pnl * 10000 # Convert to Basis Points for readability
        })

df_res_l = pd.DataFrame(results_long)
pivot_wr_l = df_res_l.pivot(index='Score_Short', columns='Score_Long', values='Win_Rate')
pivot_pnl_l = df_res_l.pivot(index='Score_Short', columns='Score_Long', values='PnL')

fig, axes = plt.subplots(1, 2, figsize=(20, 8))
sns.heatmap(pivot_wr_l, annot=True, fmt=".1%", cmap="YlGnBu", ax=axes[0])
axes[0].set_title("Dual LONG: Win Rate Sweet Spots\n(Y: Score_Short < | X: Score_Long >)")
axes[0].invert_yaxis()

sns.heatmap(pivot_pnl_l, annot=True, fmt=".1f", cmap="RdYlGn", center=0, ax=axes[1])
axes[1].set_title("Dual LONG: Avg Profit in Basis Points\n(Y: Score_Short < | X: Score_Long >)")
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig(r'C:\Users\loq\.gemini\antigravity\brain\08f08955-7051-4470-bcba-a2b3a9aebbfd\dual_long_heatmap.png')
plt.close()

# 2. SHORT STRATEGY HEATMAP
s_grid_short = np.arange(0.07, 0.12, 0.005)
l_grid_short = np.arange(-0.20, -0.09, 0.02)

results_short = []
for s_thresh in s_grid_short:
    for l_thresh in l_grid_short:
        mask = (df_test['Score_Short'] > s_thresh) & (df_test['Score_Long'] < l_thresh)
        trades = mask.sum()
        if trades >= 30:
            wr = df_test.loc[mask, 'Win_Short'].mean()
            pnl = df_test.loc[mask, 'Pnl_Short'].mean()
        else:
            wr = np.nan
            pnl = np.nan
        results_short.append({
            'Score_Short': round(s_thresh, 3),
            'Score_Long': round(l_thresh, 3),
            'Win_Rate': wr,
            'PnL': pnl * 10000 # Convert to Basis Points
        })

df_res_s = pd.DataFrame(results_short)
pivot_wr_s = df_res_s.pivot(index='Score_Long', columns='Score_Short', values='Win_Rate')
pivot_pnl_s = df_res_s.pivot(index='Score_Long', columns='Score_Short', values='PnL')

fig, axes = plt.subplots(1, 2, figsize=(20, 8))
sns.heatmap(pivot_wr_s, annot=True, fmt=".1%", cmap="YlOrRd", ax=axes[0])
axes[0].set_title("Dual SHORT: Win Rate Sweet Spots\n(Y: Score_Long < | X: Score_Short >)")
axes[0].invert_yaxis()

sns.heatmap(pivot_pnl_s, annot=True, fmt=".1f", cmap="RdYlGn", center=0, ax=axes[1])
axes[1].set_title("Dual SHORT: Avg Profit in Basis Points\n(Y: Score_Long < | X: Score_Short >)")
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig(r'C:\Users\loq\.gemini\antigravity\brain\08f08955-7051-4470-bcba-a2b3a9aebbfd\dual_short_heatmap.png')
plt.close()

print("Heatmaps saved successfully.")

Generating high-resolution heatmap for Dual-Confirmation sweet spots...


Heatmaps saved successfully.


In [29]:
import itertools
import warnings
warnings.filterwarnings('ignore')

print("="*80)
print("EXHAUSTIVE EDGE DISCOVERY - ALL POSSIBLE DUAL-MODEL COMBINATIONS")
print("="*80)

# ============================================================
# EDGE 1: Both models AGREE on LONG
# (Long model says BUY, Short model also says DON'T SHORT)
# ============================================================
print("\n--- EDGE 1: AGREEMENT LONG (Score_Long > X AND Score_Short > Y) ---")
print("(Both models agree the stock will go UP)")
results = []
for l_thresh in np.arange(0.04, 0.09, 0.005):
    for s_thresh in np.arange(0.04, 0.09, 0.005):
        mask = (df_test['Score_Long'] > l_thresh) & (df_test['Score_Short'] > s_thresh)
        trades = mask.sum()
        if trades >= 30:
            wr = df_test.loc[mask, 'Win_Long'].mean()
            pnl = df_test.loc[mask, 'Pnl_Long'].mean()
            results.append({'L(>)': round(l_thresh,3), 'S(>)': round(s_thresh,3), 'Trades': trades, 'WR': wr, 'PnL_bps': pnl*10000})
df_e1 = pd.DataFrame(results)
if not df_e1.empty:
    df_e1 = df_e1.sort_values('WR', ascending=False).head(10)
    print(df_e1.to_string(index=False, formatters={'WR': '{:.2%}'.format, 'PnL_bps': '{:.1f}'.format}))
else:
    print("No valid configurations.")

# ============================================================
# EDGE 2: Both models AGREE on SHORT  
# (Short model says SELL, Long model also says DON'T BUY)
# ============================================================
print("\n--- EDGE 2: AGREEMENT SHORT (Score_Short < X AND Score_Long < Y) ---")
print("(Both models agree the stock will go DOWN)")
results = []
for s_thresh in np.arange(-0.24, -0.08, 0.02):
    for l_thresh in np.arange(-0.24, -0.08, 0.02):
        mask = (df_test['Score_Short'] < s_thresh) & (df_test['Score_Long'] < l_thresh)
        trades = mask.sum()
        if trades >= 30:
            wr = df_test.loc[mask, 'Win_Short'].mean()
            pnl = df_test.loc[mask, 'Pnl_Short'].mean()
            results.append({'S(<)': round(s_thresh,3), 'L(<)': round(l_thresh,3), 'Trades': trades, 'WR': wr, 'PnL_bps': pnl*10000})
df_e2 = pd.DataFrame(results)
if not df_e2.empty:
    df_e2 = df_e2.sort_values('WR', ascending=False).head(10)
    print(df_e2.to_string(index=False, formatters={'WR': '{:.2%}'.format, 'PnL_bps': '{:.1f}'.format}))
else:
    print("No valid configurations.")

# ============================================================
# EDGE 3: MAXIMUM DIVERGENCE LONG 
# (Long model extremely bullish, Short model extremely bearish - already found)
# Now with FINER granularity
# ============================================================
print("\n--- EDGE 3: DIVERGENCE LONG - FINE GRAIN (Score_Long > X AND Score_Short < Y) ---")
results = []
for l_thresh in np.arange(0.065, 0.09, 0.005):
    for s_thresh in np.arange(-0.24, -0.10, 0.01):
        mask = (df_test['Score_Long'] > l_thresh) & (df_test['Score_Short'] < s_thresh)
        trades = mask.sum()
        if trades >= 30:
            wr = df_test.loc[mask, 'Win_Long'].mean()
            pnl = df_test.loc[mask, 'Pnl_Long'].mean()
            results.append({'L(>)': round(l_thresh,3), 'S(<)': round(s_thresh,3), 'Trades': trades, 'WR': wr, 'PnL_bps': pnl*10000})
df_e3 = pd.DataFrame(results)
if not df_e3.empty:
    df_e3 = df_e3.sort_values('WR', ascending=False).head(15)
    print(df_e3.to_string(index=False, formatters={'WR': '{:.2%}'.format, 'PnL_bps': '{:.1f}'.format}))

# ============================================================
# EDGE 4: MAXIMUM DIVERGENCE SHORT - FINE GRAIN
# ============================================================
print("\n--- EDGE 4: DIVERGENCE SHORT - FINE GRAIN (Score_Short > X AND Score_Long < Y) ---")
results = []
for s_thresh in np.arange(0.07, 0.12, 0.005):
    for l_thresh in np.arange(-0.20, -0.09, 0.01):
        mask = (df_test['Score_Short'] > s_thresh) & (df_test['Score_Long'] < l_thresh)
        trades = mask.sum()
        if trades >= 30:
            wr = df_test.loc[mask, 'Win_Short'].mean()
            pnl = df_test.loc[mask, 'Pnl_Short'].mean()
            results.append({'S(>)': round(s_thresh,3), 'L(<)': round(l_thresh,3), 'Trades': trades, 'WR': wr, 'PnL_bps': pnl*10000})
df_e4 = pd.DataFrame(results)
if not df_e4.empty:
    df_e4 = df_e4.sort_values('WR', ascending=False).head(15)
    print(df_e4.to_string(index=False, formatters={'WR': '{:.2%}'.format, 'PnL_bps': '{:.1f}'.format}))

# ============================================================
# EDGE 5: SCORE DIFFERENTIAL (The Spread)
# Does the RAW DIFFERENCE between the two model scores predict anything?
# ============================================================
print("\n--- EDGE 5: SCORE SPREAD ANALYSIS ---")
df_test['Score_Spread'] = df_test['Score_Long'] - df_test['Score_Short']
print("Score_Spread = Score_Long - Score_Short")
print("A large positive spread means Long model is way more bullish than Short model.")
print("A large negative spread means Short model is way more bearish than Long model.")

spread_bins = pd.qcut(df_test['Score_Spread'], q=20, duplicates='drop')
spread_analysis = df_test.groupby(spread_bins).agg(
    Count=('Score_Spread', 'size'),
    Long_WR=('Win_Long', 'mean'),
    Long_PnL=('Pnl_Long', 'mean'),
    Short_WR=('Win_Short', 'mean'),
    Short_PnL=('Pnl_Short', 'mean'),
).reset_index()
print(spread_analysis.to_string(index=False, formatters={
    'Long_WR': '{:.2%}'.format, 'Long_PnL': '{:.4%}'.format,
    'Short_WR': '{:.2%}'.format, 'Short_PnL': '{:.4%}'.format
}))

EXHAUSTIVE EDGE DISCOVERY - ALL POSSIBLE DUAL-MODEL COMBINATIONS

--- EDGE 1: AGREEMENT LONG (Score_Long > X AND Score_Short > Y) ---
(Both models agree the stock will go UP)
No valid configurations.

--- EDGE 2: AGREEMENT SHORT (Score_Short < X AND Score_Long < Y) ---
(Both models agree the stock will go DOWN)
 S(<)  L(<)  Trades     WR PnL_bps
-0.10 -0.12     524 51.15%     3.2
-0.12 -0.12     295 48.81%    -2.3
-0.10 -0.14      91 48.35%     5.2
-0.12 -0.10     894 47.43%    -4.6
-0.10 -0.10    1499 47.30%    -4.2
-0.12 -0.14      35 45.71%    -2.7
-0.14 -0.10     483 43.69%   -11.4
-0.14 -0.12     100 42.00%   -17.7
-0.16 -0.10     100 41.00%   -12.7

--- EDGE 3: DIVERGENCE LONG - FINE GRAIN (Score_Long > X AND Score_Short < Y) ---


 L(>)  S(<)  Trades     WR PnL_bps
0.085 -0.24     113 61.06%    29.0
0.080 -0.21     324 60.19%    17.9
0.080 -0.20     352 59.66%    20.1
0.080 -0.24     170 59.41%    25.2
0.075 -0.21     467 59.31%    15.2
0.080 -0.22     289 59.17%    16.6
0.080 -0.19     379 59.10%    18.8
0.085 -0.21     210 59.05%    13.7
0.075 -0.20     505 58.81%    16.4
0.075 -0.19     546 58.79%    16.2
0.075 -0.22     414 58.70%    15.6
0.085 -0.20     227 58.59%    16.6
0.075 -0.24     239 58.16%    22.0
0.085 -0.22     186 58.06%    12.8
0.080 -0.18     399 57.89%    16.3

--- EDGE 4: DIVERGENCE SHORT - FINE GRAIN (Score_Short > X AND Score_Long < Y) ---
 S(>)  L(<)  Trades     WR PnL_bps
0.115 -0.17     125 76.00%   136.6
0.110 -0.17     151 75.50%   128.6
0.105 -0.17     173 74.57%   121.9
0.115 -0.16     188 74.47%   115.8
0.110 -0.16     222 74.32%   105.9
0.100 -0.17     189 74.07%   118.9
0.115 -0.15     215 73.49%   111.3
0.115 -0.14     223 73.09%   107.1
0.105 -0.16     267 72.66%   100.3
0.100 

In [30]:
print("="*80)
print("EDGE 6: TIME-FILTERED SWEET SPOTS")
print("="*80)

# Only Hour 14 (2 PM) trades
df_h14 = df_test[df_test['Hour_Of_Day'] == 14]

print(f"\nTotal Hour-14 rows in OOS: {len(df_h14):,}")

# LONG at 2PM only
print("\n--- LONG @ 2PM ONLY: Fine grain (Score_Long > X AND Score_Short < Y) ---")
results = []
for l_thresh in np.arange(0.06, 0.09, 0.005):
    for s_thresh in np.arange(-0.24, -0.08, 0.01):
        mask = (df_h14['Score_Long'] > l_thresh) & (df_h14['Score_Short'] < s_thresh)
        trades = mask.sum()
        if trades >= 30:
            wr = df_h14.loc[mask, 'Win_Long'].mean()
            pnl = df_h14.loc[mask, 'Pnl_Long'].mean()
            results.append({'L(>)': round(l_thresh,3), 'S(<)': round(s_thresh,3), 'Trades': trades, 'WR': wr, 'PnL_bps': pnl*10000})
df_e6l = pd.DataFrame(results)
if not df_e6l.empty:
    df_e6l = df_e6l.sort_values('WR', ascending=False).head(15)
    print(df_e6l.to_string(index=False, formatters={'WR': '{:.2%}'.format, 'PnL_bps': '{:.1f}'.format}))

# SHORT at 2PM only
print("\n--- SHORT @ 2PM ONLY: Fine grain (Score_Short > X AND Score_Long < Y) ---")
results = []
for s_thresh in np.arange(0.06, 0.12, 0.005):
    for l_thresh in np.arange(-0.20, -0.08, 0.01):
        mask = (df_h14['Score_Short'] > s_thresh) & (df_h14['Score_Long'] < l_thresh)
        trades = mask.sum()
        if trades >= 30:
            wr = df_h14.loc[mask, 'Win_Short'].mean()
            pnl = df_h14.loc[mask, 'Pnl_Short'].mean()
            results.append({'S(>)': round(s_thresh,3), 'L(<)': round(l_thresh,3), 'Trades': trades, 'WR': wr, 'PnL_bps': pnl*10000})
df_e6s = pd.DataFrame(results)
if not df_e6s.empty:
    df_e6s = df_e6s.sort_values('WR', ascending=False).head(15)
    print(df_e6s.to_string(index=False, formatters={'WR': '{:.2%}'.format, 'PnL_bps': '{:.1f}'.format}))

# ============================================================
# EDGE 7: SCORE RATIO (Long / Short)
# ============================================================
print("\n" + "="*80)
print("EDGE 7: SCORE RATIO ANALYSIS (Score_Long / Score_Short)")
print("="*80)
df_test['Score_Ratio'] = df_test['Score_Long'] / df_test['Score_Short'].replace(0, np.nan)
# Filter out extreme outliers
valid_ratio = df_test[df_test['Score_Ratio'].between(-10, 10)]

ratio_bins = pd.qcut(valid_ratio['Score_Ratio'], q=20, duplicates='drop')
ratio_analysis = valid_ratio.groupby(ratio_bins).agg(
    Count=('Score_Ratio', 'size'),
    Long_WR=('Win_Long', 'mean'),
    Long_PnL=('Pnl_Long', 'mean'),
    Short_WR=('Win_Short', 'mean'),
    Short_PnL=('Pnl_Short', 'mean'),
).reset_index()
print(ratio_analysis.to_string(index=False, formatters={
    'Long_WR': '{:.2%}'.format, 'Long_PnL': '{:.4%}'.format,
    'Short_WR': '{:.2%}'.format, 'Short_PnL': '{:.4%}'.format
}))

# ============================================================
# EDGE 8: SINGLE-MODEL-ONLY @ 2PM (The Relaxed Filter)
# What if we only require ONE model but ONLY trade at 2PM?
# ============================================================
print("\n" + "="*80)
print("EDGE 8: SINGLE MODEL @ 2PM ONLY")
print("="*80)

print("\n--- Long Model ONLY @ 2PM ---")
for thresh in [0.06, 0.065, 0.07, 0.075, 0.08, 0.085]:
    mask = (df_h14['Score_Long'] > thresh)
    trades = mask.sum()
    if trades >= 30:
        wr = df_h14.loc[mask, 'Win_Long'].mean()
        pnl = df_h14.loc[mask, 'Pnl_Long'].mean()
        print(f"Score_Long > {thresh:.3f} | Trades: {trades:>5} | WR: {wr:.2%} | PnL: {pnl*10000:.1f} bps")

print("\n--- Short Model ONLY @ 2PM ---")
for thresh in [0.06, 0.07, 0.08, 0.087, 0.09, 0.10, 0.11]:
    mask = (df_h14['Score_Short'] > thresh)
    trades = mask.sum()
    if trades >= 30:
        wr = df_h14.loc[mask, 'Win_Short'].mean()
        pnl = df_h14.loc[mask, 'Pnl_Short'].mean()
        print(f"Score_Short > {thresh:.3f} | Trades: {trades:>5} | WR: {wr:.2%} | PnL: {pnl*10000:.1f} bps")

print("\n--- Inverted Long (for Shorts) @ 2PM ---")
for thresh in [-0.10, -0.12, -0.14, -0.152, -0.167, -0.18]:
    mask = (df_h14['Score_Long'] < thresh)
    trades = mask.sum()
    if trades >= 30:
        wr = df_h14.loc[mask, 'Win_Short'].mean()
        pnl = df_h14.loc[mask, 'Pnl_Short'].mean()
        print(f"Score_Long < {thresh:.3f} | Trades: {trades:>5} | WR: {wr:.2%} | PnL: {pnl*10000:.1f} bps")

EDGE 6: TIME-FILTERED SWEET SPOTS

Total Hour-14 rows in OOS: 36,972

--- LONG @ 2PM ONLY: Fine grain (Score_Long > X AND Score_Short < Y) ---
 L(>)  S(<)  Trades     WR PnL_bps
0.085 -0.24     113 61.06%    29.0
0.080 -0.21     289 60.55%    20.1
0.080 -0.24     158 60.13%    27.0
0.080 -0.20     315 60.00%    22.4
0.080 -0.22     264 59.85%    18.7
0.075 -0.21     408 59.56%    16.4
0.085 -0.21     195 59.49%    15.8
0.075 -0.22     369 59.35%    17.2
0.075 -0.20     440 59.32%    18.4
0.080 -0.19     337 59.05%    20.4
0.085 -0.20     210 59.05%    18.9
0.075 -0.19     473 58.99%    17.6
0.075 -0.24     214 58.88%    23.9
0.085 -0.22     177 58.76%    14.6
0.080 -0.18     349 58.17%    18.4

--- SHORT @ 2PM ONLY: Fine grain (Score_Short > X AND Score_Long < Y) ---
 S(>)  L(<)  Trades     WR PnL_bps
0.115 -0.17     125 76.00%   136.6
0.115 -0.16     166 75.90%   125.9
0.115 -0.15     182 75.82%   124.3
0.115 -0.14     186 75.81%   122.6
0.110 -0.16     197 75.63%   114.0
0.110 -0.17 

       Score_Ratio  Count Long_WR Long_PnL Short_WR Short_PnL
   (-10.0, -4.851]  10459  37.88% -0.1242%   43.58%  -0.0758%
  (-4.851, -3.306]  10458  37.46% -0.1201%   44.01%  -0.0799%
   (-3.306, -2.43]  10458  36.12% -0.1455%   45.19%  -0.0545%
   (-2.43, -1.763]  10458  36.95% -0.1244%   44.07%  -0.0756%
  (-1.763, -1.015]  10459  37.06% -0.1357%   43.14%  -0.0643%
   (-1.015, -0.42]  10458  38.93% -0.0817%   39.71%  -0.1183%
   (-0.42, -0.281]  10458  41.98% -0.0493%   37.47%  -0.1507%
  (-0.281, -0.209]  10458  40.64% -0.0632%   37.54%  -0.1368%
  (-0.209, -0.154]  10458  39.82% -0.0829%   38.02%  -0.1171%
 (-0.154, -0.0974]  10459  39.45% -0.0736%   37.69%  -0.1264%
(-0.0974, -0.0147]  10458  39.04% -0.0799%   38.82%  -0.1201%
  (-0.0147, 0.155]  10458  38.55% -0.0905%   39.26%  -0.1095%
    (0.155, 0.375]  10458  38.13% -0.0885%   38.45%  -0.1115%
    (0.375, 0.549]  10458  37.44% -0.0996%   40.05%  -0.1004%
    (0.549, 0.727]  10460  38.47% -0.0846%   40.52%  -0.1154%
    (0.7

In [31]:
fig, axes = plt.subplots(2, 2, figsize=(22, 16))
fig.suptitle('COMPLETE EDGE MAP: All Discovered Alpha Sources (OOS, 10bps Fee)', fontsize=16, fontweight='bold')

# ---- Panel 1: Short Model @ 2PM (The Crown Jewel) ----
ax = axes[0,0]
thresholds = [0.060, 0.070, 0.080, 0.087, 0.090, 0.100, 0.110]
trades =     [1634,  1102,  725,   577,   515,   367,   256]
wrs =        [60.47, 62.79, 65.93, 68.46, 68.16, 71.12, 73.05]
pnls =       [26.9,  39.2,  54.2,  64.0,  65.4,  84.1,  93.1]

color = plt.cm.RdYlGn(np.linspace(0.3, 0.95, len(thresholds)))
bars = ax.bar([str(t) for t in thresholds], wrs, color=color, edgecolor='black', linewidth=0.5)
for bar, t, p in zip(bars, trades, pnls):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3, 
            f'{int(t)} trades\n+{p:.0f}bps', ha='center', va='bottom', fontsize=8, fontweight='bold')
ax.axhline(y=50, color='red', linestyle='--', alpha=0.7, label='50% Break-Even')
ax.set_title('Short Model ONLY @ 2PM\n(The Crown Jewel)', fontsize=12, fontweight='bold')
ax.set_ylabel('Win Rate (%)')
ax.set_xlabel('Score_Short Threshold (>)')
ax.set_ylim(45, 80)
ax.legend()

# ---- Panel 2: Long Model @ 2PM ----
ax = axes[0,1]
thresholds_l = [0.060, 0.065, 0.070, 0.075, 0.080, 0.085]
trades_l =     [1476,  1069,  778,   579,   405,   261]
wrs_l =        [52.98, 53.04, 55.14, 56.48, 56.79, 55.56]
pnls_l =       [9.3,   10.4,  14.7,  18.0,  21.8,  20.4]

color_l = plt.cm.YlGnBu(np.linspace(0.3, 0.95, len(thresholds_l)))
bars = ax.bar([str(t) for t in thresholds_l], wrs_l, color=color_l, edgecolor='black', linewidth=0.5)
for bar, t, p in zip(bars, trades_l, pnls_l):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{int(t)} trades\n+{p:.0f}bps', ha='center', va='bottom', fontsize=8, fontweight='bold')
ax.axhline(y=50, color='red', linestyle='--', alpha=0.7, label='50% Break-Even')
ax.set_title('Long Model ONLY @ 2PM', fontsize=12, fontweight='bold')
ax.set_ylabel('Win Rate (%)')
ax.set_xlabel('Score_Long Threshold (>)')
ax.set_ylim(45, 65)
ax.legend()

# ---- Panel 3: Dual Short Lock @ 2PM (Fine Grain) ----
ax = axes[1,0]
configs = [
    ('S>.115 L<-.17', 76.0, 125, 136.6),
    ('S>.115 L<-.16', 75.9, 166, 125.9),
    ('S>.110 L<-.16', 75.6, 197, 114.0),
    ('S>.110 L<-.17', 75.5, 151, 128.6),
    ('S>.105 L<-.17', 74.6, 173, 121.9),
    ('S>.100 L<-.16', 74.4, 258, 104.7),
    ('S>.100 L<-.17', 74.1, 189, 118.9),
    ('S>.087 L<-.17', 72.1, 244, 97.1),
]
labels = [c[0] for c in configs]
wr_vals = [c[1] for c in configs]
trade_vals = [c[2] for c in configs]
pnl_vals = [c[3] for c in configs]

colors_s = plt.cm.hot_r(np.linspace(0.1, 0.6, len(configs)))
bars = ax.barh(labels, wr_vals, color=colors_s, edgecolor='black', linewidth=0.5)
for bar, t, p in zip(bars, trade_vals, pnl_vals):
    ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
            f'{t} trades | +{p:.0f}bps', va='center', fontsize=8, fontweight='bold')
ax.axvline(x=50, color='red', linestyle='--', alpha=0.7)
ax.set_title('Dual SHORT Lock @ 2PM (Fine Grain)', fontsize=12, fontweight='bold')
ax.set_xlabel('Win Rate (%)')
ax.set_xlim(60, 82)

# ---- Panel 4: Inverted Long for Shorts @ 2PM ----
ax = axes[1,1]
thresholds_inv = [-0.100, -0.120, -0.140, -0.152, -0.167]
trades_inv =     [16149,  11050,  6481,  3495,   726]
wrs_inv =        [50.80,  52.07,  53.90, 55.62,  61.98]
pnls_inv =       [-0.9,    2.2,    6.8,  13.4,   42.4]

colors_inv = plt.cm.PuBuGn(np.linspace(0.3, 0.95, len(thresholds_inv)))
bars = ax.bar([str(t) for t in thresholds_inv], wrs_inv, color=colors_inv, edgecolor='black', linewidth=0.5)
for bar, t, p in zip(bars, trades_inv, pnls_inv):
    sign = '+' if p >= 0 else ''
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{int(t)} trades\n{sign}{p:.0f}bps', ha='center', va='bottom', fontsize=8, fontweight='bold')
ax.axhline(y=50, color='red', linestyle='--', alpha=0.7, label='50% Break-Even')
ax.set_title('Inverted Long Model (for Shorts) @ 2PM', fontsize=12, fontweight='bold')
ax.set_ylabel('Win Rate (%)')
ax.set_xlabel('Score_Long Threshold (<)')
ax.set_ylim(45, 68)
ax.legend()

plt.tight_layout()
plt.savefig(r'C:\Users\loq\.gemini\antigravity\brain\08f08955-7051-4470-bcba-a2b3a9aebbfd\complete_edge_map.png', dpi=150)
plt.close()
print("Complete edge map saved.")

Complete edge map saved.
